<a href="https://colab.research.google.com/github/Bvsrkrishna/Data-Analyst-and-Engineering/blob/main/EDA3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA 3

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import sys, os
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False  #@param {type:"boolean"}
if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/clv_project")   # files survive between sessions
elif IN_COLAB:
    BASE = Path("/content/clv_project")                                                                              # lost when runtime resets
else:
    BASE = Path.cwd() / "clv_project"
BASE.mkdir(parents=True, exist_ok=True); os.chdir(BASE)
print("Working folder:", BASE)
REQUIRED = ['data/processed/eda_rfm_features.csv', 'data/processed/transactions_clean.csv']
missing = [p for p in REQUIRED if not (BASE / p).exists()]

In [ ]:
import json, warnings, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")


P = BASE / "data" / "processed"; FIG = BASE / "reports" / "figures"
def h(t): print(f"\n{'='*8} {t} {'='*8}")
rng = np.random.default_rng(42)

rfm = pd.read_csv(P / "eda_rfm_features.csv"); cu = pd.read_csv(P / "customers_clean.csv")
tx = pd.read_csv(P / "transactions_clean.csv", parse_dates=["order_date"])
d = rfm.merge(cu[["customer_id", "acquisition_channel", "country", "age"]], on="customer_id")

### 1. Distribution shape & normality

In [ ]:
h("1. Distribution shape & normality")
m = d.monetary
print(f"skew={m.skew():.2f} kurt={m.kurt():.2f}")
samp = m.sample(2000, random_state=1)
print("Shapiro raw  p=%.2e | log p=%.2e" % (stats.shapiro(samp).pvalue, stats.shapiro(np.log(samp)).pvalue))
print("D'Agostino log:", round(stats.normaltest(np.log(m)).pvalue, 4), "| KS vs lognorm:", round(stats.kstest(m, "lognorm", stats.lognorm.fit(m, floc=0)).statistic, 3))
bc, lam = stats.boxcox(m); print("Box-Cox lambda:", round(lam, 3), "skew after:", round(pd.Series(bc).skew(), 3))
fig, ax = plt.subplots(1, 2, figsize=(10, 4)); stats.probplot(np.log(m), plot=ax[0]); ax[0].set_title("Q-Q (log monetary)")
sns.ecdfplot(m, ax=ax[1]); ax[1].set_xscale("log"); fig.tight_layout(); fig.savefig(FIG / "eda_adv_qq.png", dpi=100); plt.show()

### 2. Hypothesis tests

In [ ]:
h("2. Hypothesis tests")
a = d[d.acquisition_channel == "Organic"].monetary; b = d[d.acquisition_channel == "Paid Search"].monetary
print("Welch t-test  :", stats.ttest_ind(a, b, equal_var=False)); print("Mann-Whitney  :", stats.mannwhitneyu(a, b).pvalue.round(4))
groups = [g.monetary.values for _, g in d.groupby("acquisition_channel")]
print("ANOVA p=%.4f | Kruskal p=%.4f | Levene p=%.4f" % (stats.f_oneway(*groups).pvalue, stats.kruskal(*groups).pvalue, stats.levene(*groups).pvalue))
ct = pd.crosstab(d.country, d.is_repeat); chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Chi-square repeat~country: chi2={chi2:.2f} dof={dof} p={p:.4f}")
r, p = stats.spearmanr(d.frequency, d.monetary); print(f"Spearman freq~monetary r={r:.3f} p={p:.1e}")
print("Proportion z-like (repeat Organic vs Social):", stats.fisher_exact(pd.crosstab(d.acquisition_channel.isin(["Organic"]), d.is_repeat))[1].round(4))

### 3. Effect size & multiple-testing correction

In [ ]:
h("3. Effect size & multiple-testing correction")
def cohens_d(x, y): s = np.sqrt(((len(x)-1)*x.var() + (len(y)-1)*y.var()) / (len(x)+len(y)-2)); return (x.mean() - y.mean()) / s
print("Cohen's d (Organic vs Paid):", round(cohens_d(a, b), 3))
chs = sorted(d.acquisition_channel.unique()); pv = []; pairs = []
for i in range(len(chs)):
    for j in range(i + 1, len(chs)):
        pairs.append((chs[i], chs[j])); pv.append(stats.mannwhitneyu(d[d.acquisition_channel == chs[i]].monetary, d[d.acquisition_channel == chs[j]].monetary).pvalue)
pv = np.array(pv); order = pv.argsort(); m_ = len(pv); bh = np.empty(m_); run = 1
for rank, idx in reversed(list(enumerate(order, 1))): run = min(run, pv[idx] * m_ / rank); bh[idx] = run
print("pairs significant (Bonferroni):", int((pv * m_ < .05).sum()), "| (Benjamini-Hochberg):", int((bh < .05).sum()), "of", m_)

### 4. Bootstrap confidence intervals

In [ ]:
h("4. Bootstrap confidence intervals")
def boot(x, f=np.mean, n=2000): return np.percentile([f(rng.choice(x, len(x))) for _ in range(n)], [2.5, 97.5])
print("mean spend 95% CI:", boot(m.values).round(1), "| median:", boot(m.values, np.median).round(1), "| t-interval:", np.round(stats.t.interval(.95, len(m)-1, m.mean(), stats.sem(m)), 1))

### 5. Multicollinearity - VIF (computed from R^2)

In [ ]:
h("5. Multicollinearity - VIF (computed from R^2)")
X = d[["recency", "frequency", "monetary", "aov", "tenure", "orders_per_month", "gap"]].fillna(0)
vif = {c: 1 / (1 - LinearRegression().fit(X.drop(columns=c), X[c]).score(X.drop(columns=c), X[c])) for c in X}
print({k: round(v, 1) for k, v in vif.items()}, "<- >10 = problematic")

### 6. PCA

In [ ]:
h("6. PCA")
Z = StandardScaler().fit_transform(X); pca = PCA().fit(Z)
print("explained var:", pca.explained_variance_ratio_.round(3), "| comps for 90%:", int(np.argmax(pca.explained_variance_ratio_.cumsum() >= .9)) + 1)
print(pd.DataFrame(pca.components_[:2], columns=X.columns).round(2))

### 7. Clustering (KMeans + silhouette)

In [ ]:
h("7. Clustering (KMeans + silhouette)")
Zl = StandardScaler().fit_transform(np.log1p(d[["recency", "frequency", "monetary"]]))
res = {k: round(silhouette_score(Zl, KMeans(k, n_init=5, random_state=1).fit_predict(Zl), sample_size=2000, random_state=1), 3) for k in range(2, 7)}
print("silhouette by k:", res); k = max(res, key=res.get); d["cluster"] = KMeans(k, n_init=10, random_state=1).fit_predict(Zl)
print(d.groupby("cluster").agg(n=("customer_id", "count"), recency=("recency", "mean"), freq=("frequency", "mean"), money=("monetary", "mean")).round(1))

### 8. Anomaly detection

In [ ]:
h("8. Anomaly detection")
iso = IsolationForest(contamination=.01, random_state=1).fit(Zl); d["anom"] = iso.predict(Zl) == -1
print("anomalies:", d.anom.sum(), "\n", d[d.anom].nlargest(3, "monetary")[["customer_id", "frequency", "monetary", "recency"]])

### 9. Time series: decomposition & autocorrelation (no statsmodels needed)

In [ ]:
h("9. Time series: decomposition & autocorrelation (no statsmodels needed)")
ts = tx.set_index("order_date").line_revenue.resample("MS").sum(); n = len(ts)
trend = ts.rolling(3, center=True).mean(); resid = ts - trend
print("trend slope/month: %.0f" % np.polyfit(range(n), ts.values, 1)[0], "| ACF lag1/lag2:", round(ts.autocorr(1), 2), round(ts.autocorr(2), 2))
print("residual std:", round(resid.std(), 0), "| last-3 trend:", trend.dropna().tail(3).round(0).tolist())
try:
    from statsmodels.tsa.seasonal import seasonal_decompose; print(seasonal_decompose(ts, period=12).seasonal.head(2))
except ImportError: print("statsmodels not installed -> manual decomposition used")

### 10. Pareto / concentration

In [ ]:
h("10. Pareto / concentration")
s = np.sort(m.values); cum = np.cumsum(s) / s.sum(); n = len(s); gini = 1 - 2 * np.trapz(cum, dx=1/n) if hasattr(np, "trapz") else 1 - 2 * np.trapezoid(cum, dx=1/n)
print(f"Gini={gini:.3f} | top 10% share={s[-n//10:].sum()/s.sum():.1%} | top 20% share={s[-n//5:].sum()/s.sum():.1%} | bottom 50% share={s[:n//2].sum()/s.sum():.1%}")

### 11. Feature importance & mutual information

In [ ]:
h("11. Feature importance & mutual information")
feat = ["recency", "frequency", "aov", "tenure", "orders_per_month", "gap", "age"]
Xf = d[feat].fillna(0); y = np.log1p(d.monetary)
print("MI:", {f: float(round(v, 2)) for f, v in zip(feat, mutual_info_regression(Xf, y, random_state=1))})
rf = RandomForestRegressor(150, random_state=1, n_jobs=-1).fit(Xf, y); print("RF importance:", {f: float(round(v, 2)) for f, v in zip(feat, rf.feature_importances_)})

### 12. Leakage-safe split (time-based + fit scaler on train only)

In [ ]:
h("12. Leakage-safe split (time-based + fit scaler on train only)")
cut = pd.Timestamp("2025-06-30"); tr_c = tx[tx.order_date <= cut].customer_id.unique(); te_c = tx[tx.order_date > cut].customer_id.unique()
print(f"train-period customers {len(tr_c):,} | holdout-period customers {len(te_c):,} | overlap {len(set(tr_c) & set(te_c)):,}")
print("RULE: build features only from data <= cutoff; target from data > cutoff; fit transformers on train only")

### 13. OLS with statsmodels (optional)

In [ ]:
h("13. OLS with statsmodels (optional)")
try:
    import statsmodels.api as sm; ols = sm.OLS(y, sm.add_constant(Xf)).fit(); print(ols.summary().tables[1])
except ImportError:
    from sklearn.linear_model import LinearRegression as LR; lr = LR().fit(Xf, y); print("sklearn OLS R2=%.3f coefs=%s" % (lr.score(Xf, y), {f: float(round(c, 4)) for f, c in zip(feat, lr.coef_)}))

### 14. Automated EDA summary -> JSON

In [ ]:
h("14. Automated EDA summary -> JSON")
summary = {"rows": int(len(tx)), "customers": int(d.customer_id.nunique()), "revenue": round(float(tx.line_revenue.sum()), 2),
           "repeat_rate": round(float(d.is_repeat.mean()), 3), "gini": round(float(gini), 3), "clusters": int(k),
           "anomalies": int(d.anom.sum()), "high_vif": [c for c, v in vif.items() if v > 10]}
(P / "eda_summary.json").write_text(json.dumps(summary, indent=2)); print(summary)